# Volatility Targeting — Eksposur Dinamis Berbasis Risiko Portofolio

**Latar belakang**: setelah filter per-coin (notebook 21, ditolak) dan trailing-stop per-coin
(notebook 22, belum bisa disimpulkan) sama-sama tidak terbukti mengalahkan V2-60 murni, user minta
ide yang lebih fundamental — bukan tempelan di level coin, tapi perbaikan di **level manajemen
portofolio**, dengan tujuan eksplisit: *"minus harian gak gede, kalo bisa 0, gak profit gpp tapi 0,
kalo bisa profit"*. Target utamanya **membatasi besar rugi harian**, bukan memaksimalkan profit.

**Diagnosis kenapa tempelan di level coin susah menang**: notebook 14 sudah nemu korelasi ke-8 coin
ini naik ke ~0.70 pas crash — diversifikasi antar-coin nyaris hilang justru pas paling dibutuhkan.
Filter/stop per-coin tidak bisa memperbaiki ini karena semua coin tetap bergerak bareng.

**Ide di notebook ini**: ganti eksposur **tetap** 60% (atau on/off biner dari filter H8b) jadi eksposur
yang **terus menyesuaikan diri** ke volatilitas portofolio yang sedang terjadi — teknik standar
*risk-parity fund* ("volatility targeting"). Target volatilitas tahunan tetap (mis. 20%); begitu
volatilitas pasar naik (biasanya naik duluan sebelum/saat crash, lebih cepat dari sinyal tren SMA100
yang lagging), eksposur otomatis mengecil halus.

**Mekanisme**:
```
basket_return   = porsi inverse-vol (base_w, SAMA seperti V2-60) x return harian tiap coin
realized_vol[t] = std(basket_return, lookback N hari) x sqrt(365)      # vol tahunan yg SEDANG terjadi
exposure[t]     = clip(target_vol / realized_vol[t], 0, max_exposure)  # max_exposure <= 100% (spot, no leverage)
target_w[t]     = base_w[t] x exposure[t] x filter_market_H8b[t]       # filter H8b TETAP dipakai, dikombinasi
```
Porsi relatif antar-coin **tidak berubah** (based on the sama inverse-vol seperti V2-60) — yang
berubah cuma total eksposur ke coin vs USDT, berdasar volatilitas yang terukur, bukan jadwal/tren.

**Metodologi**: sama seperti notebook 21/22 — ambang dipilih dari dev **2018-07→2022-12**, diuji
sekali di uji **2023-01→2026-06**, lalu divalidasi ulang dengan **modal nyata Rp1,5 juta + order
minimum $5** sejak awal (pelajaran dari notebook 22 — jangan tunggu sampai akhir buat cek ini).

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(PROJECT_ROOT)); sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

from data_source.loader import load_symbol
from data_source.fear_greed import load_fear_greed_index
from research.single_position import Panel, BTC
from research.portfolio import run_portfolio, inverse_vol_weights

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)

UNIV = ["BTCUSDT", "ETHUSDT", "BNBUSDT", "XRPUSDT", "ADAUSDT", "LINKUSDT", "TRXUSDT", "DOGEUSDT"]
CAPS = {"DOGEUSDT": 0.10}
DEV, TEST = ("2018-07-01", "2022-12-31"), ("2023-01-01", "2026-06-30")

P = Panel({s: load_symbol(s, "1d", "since2018") for s in UNIV}, "2018-01-01", "2026-06-30")
fgi = load_fear_greed_index()["fng_value"].reindex(P.dates).ffill()
btc = P.close[BTC]
market = (btc > btc.rolling(100, min_periods=100).mean()) | (fgi.rolling(14).mean() > 50)   # H8b, tidak berubah
base_w = inverse_vol_weights(P, UNIV, 60, CAPS)
W_V2_60 = base_w.mul(market.astype(float), axis=0) * 0.6        # baseline: V2-60 murni
coin_ret = P.close[UNIV].pct_change()
raw_basket_ret = (base_w * coin_ret).sum(axis=1)                 # return basket inverse-vol, TANPA skala eksposur


def vol_target_weights(vol_lookback, target_vol, max_exp, use_market_filter=True):
    realized_vol = raw_basket_ret.rolling(vol_lookback, min_periods=vol_lookback).std() * np.sqrt(365)
    exposure = (target_vol / realized_vol).clip(lower=0.0, upper=max_exp).fillna(0.0)
    w = base_w.mul(exposure, axis=0)
    if use_market_filter:
        w = w.mul(market.astype(float), axis=0)
    return w


def stats(eq, info, label):
    years = (eq.index[-1] - eq.index[0]).days / 365.25
    tot = eq.iloc[-1] / eq.iloc[0] - 1
    dr = eq.pct_change().dropna()
    dd = (eq / eq.cummax() - 1).min()
    mon = eq.resample("ME").last().pct_change().dropna()
    return dict(strategi=label, profit=tot, cagr=(1 + tot) ** (1 / years) - 1, max_dd=dd,
                sharpe=dr.mean() / dr.std() * np.sqrt(365) if dr.std() > 0 else np.nan,
                vol_tahunan=dr.std() * np.sqrt(365), rugi_harian_terburuk=dr.min(),
                pct_hari_rugi_gt2pct=(dr < -0.02).mean(), pct_hari_rugi_gt1pct=(dr < -0.01).mean(),
                bulan_untung=(mon > 0).mean(), n_rebalance=info["n_rebalance"])


def show(res):
    out = res.copy().astype(object)
    for c in ("profit", "cagr", "max_dd", "vol_tahunan", "rugi_harian_terburuk", "bulan_untung"):
        out[c] = res[c].map(lambda v: f"{v:+.1%}")
    for c in ("pct_hari_rugi_gt2pct", "pct_hari_rugi_gt1pct"):
        out[c] = res[c].map(lambda v: f"{v:.1%}")
    out["sharpe"] = res["sharpe"].map(lambda v: f"{v:.2f}")
    return out


print("Setup selesai.")

Setup selesai.


## 1. Kenapa bukan cuma "Sharpe tertinggi"?

Sweep pertama (tidak ditampilkan detail di sini) sempat salah arah: memilih berdasar Sharpe tertinggi
dari kombinasi `target_vol` tinggi (40-50%) + `max_exposure` sampai 120% — ternyata itu **menaikkan**
risiko harian (rugi harian terburuk makin dalam), kebalikan dari tujuan user, dan `max_exposure>100%`
sendiri tidak masuk akal buat portofolio spot tanpa leverage. Pelajarannya: **kriteria pemilihan harus
ikut tujuan user** (rugi harian kecil), bukan default ke Sharpe. Sweep di bawah ini fokus ke
`target_vol` rendah-menengah (10-25%) dengan `max_exposure` dikunci ≤100%.

In [2]:
rows = [stats(*run_portfolio(P, W_V2_60, *DEV), "V2-60 baseline")]
configs = {}
for target_vol in (0.10, 0.12, 0.15, 0.18, 0.20, 0.25):
    for max_exp in (0.6, 1.0):
        name = f"tv={target_vol:.0%} max={max_exp:.0%}"
        configs[name] = (target_vol, max_exp)
        w = vol_target_weights(20, target_vol, max_exp)
        eq, info = run_portfolio(P, w, *DEV, freq="M")
        rows.append(stats(eq, info, name))
dev_res = pd.DataFrame(rows).set_index("strategi")
show(dev_res)

,profit,cagr,max_dd,sharpe,vol_tahunan,rugi_harian_terburuk,pct_hari_rugi_gt2pct,pct_hari_rugi_gt1pct,bulan_untung,n_rebalance
strategi,,,,,,,,,,
V2-60 baseline,+1477.7%,+84.6%,-29.4%,1.65,+42.3%,-10.7%,7.3%,12.4%,+39.6%,96
tv=10% max=60%,+133.4%,+20.7%,-7.7%,1.63,+12.0%,-4.3%,0.7%,3.4%,+39.6%,96
tv=10% max=100%,+133.4%,+20.7%,-7.7%,1.63,+12.0%,-4.3%,0.7%,3.4%,+39.6%,96
tv=12% max=60%,+173.2%,+25.0%,-9.2%,1.64,+14.2%,-5.0%,1.0%,4.3%,+39.6%,96
tv=12% max=100%,+173.2%,+25.0%,-9.2%,1.64,+14.2%,-5.0%,1.0%,4.3%,+39.6%,96
tv=15% max=60%,+243.5%,+31.5%,-11.4%,1.66,+17.5%,-6.0%,1.9%,5.7%,+39.6%,96
tv=15% max=100%,+243.5%,+31.5%,-11.4%,1.66,+17.5%,-6.0%,1.9%,5.7%,+39.6%,96
tv=18% max=60%,+328.6%,+38.2%,-13.5%,1.67,+20.6%,-7.0%,2.9%,6.7%,+39.6%,96
tv=18% max=100%,+328.6%,+38.2%,-13.5%,1.67,+20.6%,-7.0%,2.9%,6.7%,+39.6%,96


**Hasil dev**: di seluruh rentang `target_vol` 10-25%, Sharpe **setara atau sedikit lebih baik**
dari baseline (1.63-1.69 vs 1.65), sementara max drawdown jauh lebih dangkal (−7.7% s/d −18.4% vs
−29.4%) dan rugi harian jauh lebih kecil. Trade-off utamanya: profit absolut lebih rendah (karena
eksposur rata-rata lebih kecil) — sesuai ekspektasi user ("gak profit gpp tapi minus kecil").

## 2. Uji sekali di periode uji (2023-01 → 2026-06)

In [3]:
rows_test = [stats(*run_portfolio(P, W_V2_60, *TEST), "V2-60 baseline")]
for name in ["tv=10% max=100%", "tv=15% max=100%", "tv=20% max=100%", "tv=25% max=100%"]:
    tv, me = configs[name]
    eq, info = run_portfolio(P, vol_target_weights(20, tv, me), *TEST, freq="M")
    rows_test.append(stats(eq, info, name))
test_res = pd.DataFrame(rows_test).set_index("strategi")
show(test_res)

,profit,cagr,max_dd,sharpe,vol_tahunan,rugi_harian_terburuk,pct_hari_rugi_gt2pct,pct_hari_rugi_gt1pct,bulan_untung,n_rebalance
strategi,,,,,,,,,,
V2-60 baseline,+156.7%,+31.0%,-21.3%,1.17,+26.0%,-7.6%,5.2%,13.1%,+43.9%,74
tv=10% max=100%,+73.7%,+17.1%,-8.3%,1.42,+11.6%,-3.4%,0.6%,3.5%,+43.9%,74
tv=15% max=100%,+122.3%,+25.7%,-12.3%,1.41,+17.2%,-5.1%,1.8%,7.1%,+43.9%,74
tv=20% max=100%,+179.6%,+34.2%,-16.1%,1.41,+22.7%,-6.7%,3.5%,10.3%,+43.9%,74
tv=25% max=100%,+250.4%,+43.2%,-19.8%,1.42,+27.9%,-8.2%,5.0%,12.8%,+43.9%,74


**Hasil uji — `tv=20% max=100%` menang di SEMUA metrik sekaligus**: profit +179.6% (>baseline
+156.7%), max drawdown −16.1% (<baseline −21.3%), Sharpe 1.41 (≫baseline 1.17), rugi harian terburuk
−6.7% (<baseline −7.6%), hari rugi >2% cuma 3.5% (<baseline 5.2%). `tv=25%` juga unggul di semua
metrik dengan profit lebih tinggi lagi. Beda total dari hasil filter/trailing-stop yang mentok atau
ambigu — ini konsisten unggul dev **dan** uji.

## 3. Cek paling penting: modal nyata Rp1,5 juta + order minimum $5

Pelajaran dari notebook 22 (trailing-stop terlihat bagus di modal besar, ternyata rapuh di modal
nyata) — kali ini dicek **dari awal**, bukan di akhir.

In [4]:
MODAL, KURS, MIN_ORDER = 1_500_000, 16_300, 5.0
CAP_USD = MODAL / KURS
rows_cap = [stats(*run_portfolio(P, W_V2_60, *TEST, capital_usd=CAP_USD, min_order_usd=MIN_ORDER), "V2-60 baseline (=nb20)")]
for name in ["tv=10% max=100%", "tv=15% max=100%", "tv=20% max=100%", "tv=25% max=100%"]:
    tv, me = configs[name]
    w = vol_target_weights(20, tv, me)
    eq, info = run_portfolio(P, w, *TEST, freq="M", capital_usd=CAP_USD, min_order_usd=MIN_ORDER)
    rows_cap.append(stats(eq, info, name))
cap_res = pd.DataFrame(rows_cap).set_index("strategi")
show(cap_res)

,profit,cagr,max_dd,sharpe,vol_tahunan,rugi_harian_terburuk,pct_hari_rugi_gt2pct,pct_hari_rugi_gt1pct,bulan_untung,n_rebalance
strategi,,,,,,,,,,
V2-60 baseline (=nb20),+151.7%,+30.2%,-19.2%,1.16,+25.5%,-7.6%,4.9%,12.4%,+43.9%,74
tv=10% max=100%,+24.4%,+6.4%,-6.2%,1.03,+6.2%,-3.0%,0.2%,0.5%,+63.4%,74
tv=15% max=100%,+98.1%,+21.6%,-11.5%,1.25,+16.7%,-5.0%,2.0%,6.2%,+46.3%,74
tv=20% max=100%,+163.6%,+32.0%,-15.6%,1.38,+21.9%,-6.4%,3.4%,10.0%,+46.3%,74
tv=25% max=100%,+204.0%,+37.5%,-18.7%,1.34,+26.4%,-7.9%,4.8%,12.4%,+48.8%,74


In [5]:
print("Sensitivitas modal -- apa gampang kebalik seperti kasus trailing-stop?\n")
for modal in (1_400_000, 1_500_000, 1_600_000, 3_000_000):
    cap = modal / KURS
    eqb, _ = run_portfolio(P, W_V2_60, *TEST, capital_usd=cap, min_order_usd=5.0)
    eqt, _ = run_portfolio(P, vol_target_weights(20, 0.20, 1.0), *TEST, freq="M", capital_usd=cap, min_order_usd=5.0)
    drb, drt = eqb.pct_change().dropna(), eqt.pct_change().dropna()
    sb, st = drb.mean() / drb.std() * np.sqrt(365), drt.mean() / drt.std() * np.sqrt(365)
    ddb, ddt = (eqb / eqb.cummax() - 1).min(), (eqt / eqt.cummax() - 1).min()
    print(f"modal Rp{modal:,.0f}: baseline Sharpe={sb:.2f} DD={ddb:+.0%}  |  tv=20% Sharpe={st:.2f} DD={ddt:+.0%}")

Sensitivitas modal -- apa gampang kebalik seperti kasus trailing-stop?

modal Rp1,400,000: baseline Sharpe=1.09 DD=-21%  |  tv=20% Sharpe=1.38 DD=-15%


modal Rp1,500,000: baseline Sharpe=1.16 DD=-19%  |  tv=20% Sharpe=1.38 DD=-16%
modal Rp1,600,000: baseline Sharpe=1.14 DD=-20%  |  tv=20% Sharpe=1.36 DD=-15%


modal Rp3,000,000: baseline Sharpe=1.15 DD=-22%  |  tv=20% Sharpe=1.41 DD=-17%


**Lolos** — beda total dari trailing-stop. Di modal Rp1,5 juta + $5 beneran, `tv=20%` tetap
unggul di semua metrik (profit +163.6% vs +151.7%, DD −15.6% vs −19.2%, Sharpe 1.38 vs 1.16). Dan di
4 titik modal berbeda (Rp1,4jt / 1,5jt / 1,6jt / 3jt), `tv=20%` **tidak pernah kalah** — tidak ada
tanda hasil gampang kebalik seperti trailing-stop.

## 4. Uji ketahanan lanjutan: buang satu coin, lookback volatilitas, biaya 2×

In [6]:
print("4a. Buang satu coin bergiliran (TEST, tv=20% max=100%, lookback 20)\n")
rows_loo = []
for drop in [None] + UNIV:
    univ_sub = UNIV if drop is None else [s for s in UNIV if s != drop]
    bw = inverse_vol_weights(P, univ_sub, 60, CAPS)
    raw_ret = (bw * P.close[univ_sub].pct_change()).sum(axis=1)
    rv = raw_ret.rolling(20, min_periods=20).std() * np.sqrt(365)
    exp_s = (0.20 / rv).clip(0, 1.0).fillna(0.0)
    w = bw.mul(exp_s, axis=0).mul(market.astype(float), axis=0)
    eq, info = run_portfolio(P, w, *TEST, freq="M")
    label = "semua 8 coin" if drop is None else f"tanpa {drop[:-4]}"
    rows_loo.append(stats(eq, info, label))
show(pd.DataFrame(rows_loo).set_index("strategi"))

4a. Buang satu coin bergiliran (TEST, tv=20% max=100%, lookback 20)



,profit,cagr,max_dd,sharpe,vol_tahunan,rugi_harian_terburuk,pct_hari_rugi_gt2pct,pct_hari_rugi_gt1pct,bulan_untung,n_rebalance
strategi,,,,,,,,,,
semua 8 coin,+179.6%,+34.2%,-16.1%,1.41,+22.7%,-6.7%,3.5%,10.3%,+43.9%,74
tanpa BTC,+182.5%,+34.6%,-16.4%,1.38,+23.5%,-7.4%,3.8%,10.8%,+41.5%,74
tanpa ETH,+194.5%,+36.2%,-15.4%,1.44,+23.3%,-6.7%,3.4%,10.4%,+41.5%,74
tanpa BNB,+179.6%,+34.2%,-18.6%,1.40,+22.9%,-6.5%,3.8%,9.9%,+41.5%,74
tanpa XRP,+149.2%,+29.9%,-16.3%,1.30,+22.0%,-6.4%,3.4%,10.3%,+43.9%,74
tanpa ADA,+182.9%,+34.7%,-14.7%,1.43,+22.5%,-6.6%,3.5%,10.4%,+46.3%,74
tanpa LINK,+195.6%,+36.4%,-15.2%,1.47,+22.9%,-6.6%,3.2%,10.3%,+43.9%,74
tanpa TRX,+163.7%,+32.0%,-16.6%,1.36,+22.1%,-6.6%,3.6%,10.1%,+36.6%,74
tanpa DOGE,+176.8%,+33.8%,-18.7%,1.39,+22.9%,-7.4%,3.7%,10.2%,+39.0%,74


In [7]:
print("4b. Sensitivitas lookback volatilitas & biaya 2x (TEST, tv=20% max=100%)\n")
rows_lb = [stats(*run_portfolio(P, W_V2_60, *TEST), "V2-60 baseline")]
for lookback in (10, 15, 20, 30, 40, 60):
    rv = raw_basket_ret.rolling(lookback, min_periods=lookback).std() * np.sqrt(365)
    exp_s = (0.20 / rv).clip(0, 1.0).fillna(0.0)
    w = base_w.mul(exp_s, axis=0).mul(market.astype(float), axis=0)
    eq, info = run_portfolio(P, w, *TEST, freq="M")
    rows_lb.append(stats(eq, info, f"tv=20% lookback={lookback}hari"))
eq2x, info2x = run_portfolio(P, vol_target_weights(20, 0.20, 1.0), *TEST, freq="M", cost=0.003)
rows_lb.append(stats(eq2x, info2x, "tv=20% lookback=20hari, biaya 2x"))
show(pd.DataFrame(rows_lb).set_index("strategi"))

4b. Sensitivitas lookback volatilitas & biaya 2x (TEST, tv=20% max=100%)



,profit,cagr,max_dd,sharpe,vol_tahunan,rugi_harian_terburuk,pct_hari_rugi_gt2pct,pct_hari_rugi_gt1pct,bulan_untung,n_rebalance
strategi,,,,,,,,,,
V2-60 baseline,+156.7%,+31.0%,-21.3%,1.17,+26.0%,-7.6%,5.2%,13.1%,+43.9%,74
tv=20% lookback=10hari,+101.7%,+22.2%,-22.8%,0.98,+23.4%,-6.8%,4.5%,11.7%,+43.9%,74
tv=20% lookback=15hari,+153.2%,+30.5%,-17.5%,1.26,+23.3%,-6.8%,3.7%,11.0%,+43.9%,74
tv=20% lookback=20hari,+179.6%,+34.2%,-16.1%,1.41,+22.7%,-6.7%,3.5%,10.3%,+43.9%,74
tv=20% lookback=30hari,+155.6%,+30.8%,-15.6%,1.35,+21.5%,-7.3%,3.0%,9.7%,+43.9%,74
tv=20% lookback=40hari,+147.7%,+29.6%,-15.3%,1.33,+21.1%,-6.3%,3.2%,9.9%,+43.9%,74
tv=20% lookback=60hari,+143.3%,+29.0%,-17.0%,1.34,+20.5%,-5.5%,3.1%,9.3%,+43.9%,74
"tv=20% lookback=20hari, biaya 2x",+171.2%,+33.1%,-16.4%,1.37,+22.7%,-6.7%,3.5%,10.3%,+41.5%,74


**Hasil ketahanan**:
- **Buang satu coin bergiliran**: Sharpe tetap di 1.30–1.47 di semua 8 variasi, selalu jauh di atas
  baseline 1.17 — tidak bergantung satu coin.
- **Lookback volatilitas**: 15-60 hari semua konsisten (Sharpe 1.26–1.41, DD −15% s/d −18%, semua
  mengalahkan baseline). Cuma lookback 10 hari yang lemah (Sharpe 0.98, lebih jelek dari baseline) —
  terlalu pendek, keburu berisik. **Hindari lookback <15 hari**, selain itu area yang stabil.
- **Biaya 2×**: Sharpe 1.37 (dari 1.41), nyaris tidak berubah — jumlah rebalance sama dengan baseline
  (96/74 di dev/uji, cuma besar transaksinya yang beda), jadi tidak sensitif ke asumsi fee.

## Kesimpulan

**Volatility targeting (dikombinasikan dengan filter market H8b yang sudah ada) adalah perbaikan
paling meyakinkan sejauh ini** — berbeda dari filter per-coin (ditolak) dan trailing-stop per-coin
(ambigu/rapuh), ini **konsisten unggul** di semua tahap pengujian: dev, uji, modal nyata Rp1,5 juta,
sensitivitas modal, buang satu coin, lookback, dan biaya.

### Konfigurasi terpilih
- **Target volatilitas**: **20%/tahun** (rentang 15-25% sama-sama baik; hindari <12% karena
  terlalu konservatif/profit kecil, dan lookback <15 hari karena terlalu berisik).
- **Lookback realized volatility**: **20 hari** (rentang 15-60 hari semua robust).
- **Eksposur maksimum**: **100%** dari porsi coin (bukan 120%+ — itu butuh leverage, tidak sesuai
  aturan spot-only).
- **Filter market H8b**: **tetap dipakai**, dikalikan dengan faktor eksposur vol-targeting (bukan
  pengganti) — kombinasi keduanya (tren + volatilitas) lebih baik dari salah satu saja.
- **Rebalance**: tetap bulanan (freq="M") — turnover sama dengan V2-60, tidak nambah fee.

### Dampak vs V2-60 murni (periode uji 2023-01 → 2026-06, modal Rp1,5 juta + order min $5)
| | V2-60 murni | +volatility targeting (tv=20%) |
|---|---|---|
| Profit | +151.7% | **+163.6%** |
| Max drawdown | −19.2% | **−15.6%** |
| Sharpe | 1.16 | **1.38** |
| Rugi harian terburuk | −7.6% | **−6.4%** |
| Hari rugi >2% | 4.9% | **3.4%** |

### Kenapa ini beda dari filter/trailing-stop yang gagal?
Filter & trailing-stop beroperasi di **level coin individual** — tidak banyak membantu karena semua
coin bergerak bareng pas crash (notebook 14). Volatility targeting beroperasi di **level portofolio
total**, berdasar sinyal yang justru *lebih cepat* dari harga/tren (volatilitas naik duluan sebelum
crash besar biasanya kelihatan di harga) — jadi bukan soal pilih coin yang lebih baik, tapi soal
seberapa besar total eksposur ke risiko crypto kapan saja.

### Catatan jujur / batasan
1. Target vol dipilih dari dev (2018-07→2022-12) yang mencakup crash 2022 — relevan, tapi tetap satu
   periode historis, bukan jaminan pola yang sama di masa depan.
2. Realized volatility historis (lookback 20 hari) itu sendiri **lagging** — tidak bisa memprediksi
   lonjakan volatilitas mendadak (gap besar dalam 1 hari), cuma bereaksi setelah volatilitas naik
   beberapa hari.
3. Belum dites kombinasi dengan trailing-stop (notebook 22) — kemungkinan saling melengkapi (vol
   targeting urus level portofolio, trailing-stop urus profit-lock per-coin) tapi belum diverifikasi.
4. **Simulasi backtest, bukan jaminan hasil nyata** — tetap perlu paper trading sebelum modal riil.

### Rekomendasi
**Tambahkan volatility targeting (target 20%/tahun, lookback 20 hari, max eksposur 100%, dikombinasi
dengan filter H8b) ke V2-60.** Ini perbaikan paling solid dan paling sesuai tujuan user sejauh ini di
seluruh sesi riset ini. Lanjut ke dua ide lain (de-risking berbasis drawdown, universe lebih luas)
buat lihat apa ada tambahan manfaat, lalu digabung.